# Econ 148 Track B Reproduction Notebook

Michael Thaler (2024), *The Fake News Effect: Experimentally Identifying Motivated Reasoning Using Trust in News*. This notebook is a Python/Colab translation of the central Stata replication workflow.

In [ ]:
import os, sys
from pathlib import Path

def find_project_root(start):
    start = Path(start).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "analysis.py").exists() and (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find project root containing src/analysis.py and data/")

ROOT = find_project_root(Path.cwd())
os.chdir(ROOT)

print("Working directory:", ROOT)
print("analysis.py exists:", (ROOT / "src" / "analysis.py").exists())
print("Data files:", [p.name for p in (ROOT / "data").glob("*.csv")])

## 1. Load helper functions and data

The analysis script creates summary statistics, figures, and fixed-effects regressions. It is kept as a separate `.py` file so the same code can be run in Colab or from the command line.

In [ ]:
sys.path.insert(0, str(ROOT/'src'))
import analysis

df, rep = analysis.load_data()
df = analysis.add_vars(df)
rep = analysis.add_vars(rep)
print(df.shape)
df[['code','topic_id','round_number','prob_true','pro_party','anti_party','neutral_news']].head()

## 2. Summary statistics

In [ ]:
summary = analysis.make_summary_stats(df)
summary

## 3. Recreate core figures

These are Python recreations of the main CDF and bar charts used to show politically motivated reasoning.

In [ ]:
analysis.make_figures(df)
from IPython.display import Image, display
for fig in ['figure1_cdf_pro_antiparty.png','figure2_motivated_reasoning_partisanship.png','figure3_fake_true_news.png','figure4_by_topic.png']:
    print(fig)
    display(Image(filename=str(ROOT/'figures'/fig)))

## 4. Main fixed-effects regressions

The Python function residualizes variables by subject, topic, and round fixed effects, then estimates OLS with subject-clustered standard errors. This reproduces the core `reghdfe` logic without requiring Stata.

In [ ]:
regs = analysis.make_regression_tables(df)
regs

## 5. Replication experiment check

In [ ]:
drepl = rep[(rep['net_party'] != 0) & ((rep['pro_party']+rep['anti_party'])==1) & rep['prob_true'].notna()].copy()
res, n = analysis.cluster_reg('prob_true ~ pro_party + C(code) + C(topic_id) + C(round_number)', drepl)
print('Replication sample N:', n)
print('Pro-party coefficient:', res.params['pro_party'])
print('Clustered SE:', res.bse['pro_party'])
print('p-value:', res.pvalues['pro_party'])

## 6. Export all outputs

In [ ]:
# This cell regenerates every output used by the report.
analysis.main()
print('Figures written to:', ROOT/'figures')
print('Tables written to:', ROOT/'tables')

## Interpretation

The central result is replicated: pro-party news is rated as more likely to be true than anti-party news. The reproduced fixed-effects coefficient is about 0.088, very close to the published main specification. The other signs match the paper: stronger effect among partisans, pro-party news above neutral news, anti-party news below neutral news, and lower ratings for actual True News than Fake News on politicized questions because Fake News reinforces subjects’ motivated errors.